<a href="https://colab.research.google.com/github/ArtSharan/SDC_CODES/blob/main/Resume_Analyzer_with_Al_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai langchain gradio PyPDF2
import openai
import gradio as gr
import PyPDF2
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# ====== Set OpenAI API Key ======
openai.api_key = "your-openai-api-key"  # Replace with your OpenAI API key

# ====== Define a Prompt for Resume Analysis ======
prompt_template = PromptTemplate(
    input_variables=["resume_text"],
    template="""
    You are an AI that evaluates resumes. Please analyze the following resume and provide insights and suggestions to improve it:

    - Check for proper formatting, structure, and alignment of skills, experience, and education.
    - Suggest improvements for making the resume more attractive and effective for recruiters.
    - Highlight any missing sections or critical keywords that could be added.

    Resume: {resume_text}
    """
)

# ====== Initialize OpenAI LLM ======
llm = OpenAI(temperature=0.5)

# ====== Create a Resume Analysis Chain ======
analysis_chain = LLMChain(llm=llm, prompt=prompt_template)

# ====== Resume Analysis Function ======
def analyze_resume(resume_text):
    analysis = analysis_chain.run(resume_text)
    return analysis

# ====== Extract Text from PDF Function ======
def extract_text_from_pdf(pdf_file):
    with open(pdf_file, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
    return text

# ====== Gradio Interface for Uploading and Analyzing Resumes ======
def analyze_uploaded_resume(resume_file):
    if resume_file.name.endswith(".pdf"):
        resume_text = extract_text_from_pdf(resume_file.name)
    else:
        resume_text = resume_file.read().decode("utf-8")

    # Analyze the resume and provide feedback
    analysis = analyze_resume(resume_text)
    return analysis

# Create a simple Gradio interface
gr.Interface(
    fn=analyze_uploaded_resume,
    inputs=gr.File(type="file", label="Upload Resume (PDF or Text)"),
    outputs="text",
    title="AI Resume Analyzer",
    description="Upload your resume in PDF or text format to get personalized suggestions and insights for improvement.",
).launch()